# W1 · K7 + Esc + K25 — the checks only a person at the game can do

**K7:** do a favorite's shortcut paste into the game's chat box? (the game or its anti-cheat may ignore the synthetic Ctrl+V).
**K25:** everything left in `.memory/active-issues/unverified-on-windows.md` — **read from that file when you run this**,
so it is never out of date; each numbered `(1) (2) ...` check there is one prompt. That includes the 7 **Esc** checks of
the popup windows. The three jobs with their own notebook (updater, firewall, interface) are left out here.

Run the **dev build of `main`** (`cargo tauri dev` as Administrator) with the game running. Answer each prompt
`pass`, `fail: what you saw`, or `skip`.

In [ ]:
import sys
from pathlib import Path
here = Path.cwd().resolve()              # Jupyter starts a notebook in its own folder:
root = next(p for p in [here, *here.parents] if (p / "w1" / "common.py").exists())
sys.path.insert(0, str(root))            # ...so find test-w1 above it
from w1 import common
from w1.common import Recorder, capture, ask_text
from w1 import checklists
rec = Recorder("w1-checklists")
if not common.require_windows_admin(rec):
    raise SystemExit("Start Jupyter from an Administrator terminal, then run again.")

## K7 · Favorite shortcuts paste into the game

In [ ]:
for item in checklists.K7:
    rec.manual(item.check, item.title, item.steps, item.expect)

## K25 · The rest of the unverified list
The next cell reads the file and lists its sections. Pick the ones to run (the default is all).
Tip: the newest sections are at the bottom (the popup windows, the Esc checks).

In [ ]:
FILE = common.ROOT.parent / ".memory" / "active-issues" / "unverified-on-windows.md"
items = checklists.plan(checklists.parse_bullets(FILE.read_text(encoding="utf-8")))
sections = list(dict.fromkeys(i.section for i in items))
for n, s in enumerate(sections, 1):
    print(f"{n:2}. {s}  ({sum(1 for i in items if i.section == s)} checks)")
chosen = checklists.parse_selection(ask_text("SECTIONS", "Sections to run, e.g. 24-28 or 3,7 [all]: "), len(sections))
print("running:", [sections[i] for i in chosen])

In [ ]:
last = None
for i in chosen:
    for item in (x for x in items if x.section == sections[i]):
        if item.section != last:
            print("\n" + "#" * 70 + f"\n# {item.section}\n# {item.preface}\n" + "#" * 70)
            last = item.section
        rec.manual(item.check, item.title, item.steps, item.expect)

## Report
Paste the output of the next cell to Claude.

In [ ]:
print(rec.report())